# Loading the Billeh V1 Network

Learn how to load and explore the Allen Institute V1 network data.

## Table of Contents
1. [Billeh Network Overview](#1-billeh-network-overview)
2. [Data Directory Structure](#2-data-directory-structure)
3. [load_billeh() Function](#3-load_billeh-function)
4. [Neuron Parameters (node_params)](#4-neuron-parameters-node_params)
5. [Neuron Type Statistics](#5-neuron-type-statistics)
6. [Input Population](#6-input-population)
7. [Data Validation](#7-data-validation)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import Counter
import pandas as pd

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

In [ ]:
# Configuration - MODIFY THIS PATH
DATA_DIR = Path("/path/to/GLIF_network")  # <-- CHANGE THIS

# Verify data exists
required_files = {
    "network_dat.pkl": DATA_DIR / "network_dat.pkl",
    "input_dat.pkl": DATA_DIR / "input_dat.pkl",
    "v1_nodes.h5": DATA_DIR / "network" / "v1_nodes.h5",
    "v1_node_types.csv": DATA_DIR / "network" / "v1_node_types.csv",
}

print("Checking required files:")
all_found = True
for name, path in required_files.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  OK: {name} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {name}")
        all_found = False

if not all_found:
    print("\nPlease update DATA_DIR to point to your GLIF_network directory.")

## 1. Billeh Network Overview

The **Billeh V1 Model** is a biologically detailed model of mouse primary visual cortex (V1), developed by the Allen Institute for Brain Science.

### Key Statistics

| Property | Value |
|----------|-------|
| Total neurons | ~230,000 |
| Core neurons (400um radius) | ~51,316 |
| Neuron types | 111 |
| Synapses | ~300 million |
| Cortical layers | L1, L2/3, L4, L5, L6 |
| E/I ratio | ~80/20 |

### Model Features

- **GLIF3 neurons**: Generalized Leaky Integrate-and-Fire with adaptation
- **Biophysically constrained**: Parameters from experimental recordings
- **Layer-specific connectivity**: Different connection patterns per layer
- **Receptor types**: AMPA, NMDA, GABA_A, GABA_B

## 2. Data Directory Structure

The GLIF_network directory contains:

```
GLIF_network/
├── network_dat.pkl        # Recurrent connectivity (pickled dict)
├── input_dat.pkl          # LGN input connectivity (pickled dict)
└── network/
    ├── v1_nodes.h5        # Neuron positions (HDF5 format)
    └── v1_node_types.csv  # Neuron type definitions
```

In [ ]:
# Explore v1_node_types.csv
try:
    node_types_path = DATA_DIR / "network" / "v1_node_types.csv"
    df = pd.read_csv(node_types_path, delimiter=' ')
    
    print(f"Node types CSV: {len(df)} neuron types")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 10 entries:")
    print(df.head(10))
except FileNotFoundError:
    print("v1_node_types.csv not found. Update DATA_DIR path.")

In [ ]:
# Explore v1_nodes.h5
try:
    import h5py
    
    h5_path = DATA_DIR / "network" / "v1_nodes.h5"
    with h5py.File(h5_path, 'r') as f:
        print("HDF5 structure:")
        
        def print_structure(name, obj):
            indent = "  " * name.count('/')
            if isinstance(obj, h5py.Group):
                print(f"{indent}{name}/")
            else:
                print(f"{indent}{name}: shape={obj.shape}, dtype={obj.dtype}")
        
        f.visititems(print_structure)
        
        # Sample position data
        x = np.array(f['nodes']['v1']['0']['x'])
        y = np.array(f['nodes']['v1']['0']['y'])
        z = np.array(f['nodes']['v1']['0']['z'])
        
        print(f"\nTotal neurons in HDF5: {len(x):,}")
        print(f"Position ranges:")
        print(f"  x: [{x.min():.1f}, {x.max():.1f}]")
        print(f"  y: [{y.min():.1f}, {y.max():.1f}]")
        print(f"  z: [{z.min():.1f}, {z.max():.1f}]")
        
except FileNotFoundError:
    print("v1_nodes.h5 not found. Update DATA_DIR path.")

## 3. load_billeh() Function

The main entry point for loading network data.

In [ ]:
from v1_jax.data import load_billeh

# Function signature
print("load_billeh() Parameters:")
print("=" * 60)
params = {
    "n_input": "Number of LGN input neurons (default: 17400)",
    "n_neurons": "Number of V1 neurons to select",
    "core_only": "Only use neurons within 400um radius",
    "data_dir": "Path to GLIF_network directory",
    "seed": "Random seed for neuron selection",
    "connected_selection": "Select neurons by connectivity (closest to center)",
    "use_dale_law": "Enforce Dale's law on weights",
}

for param, desc in params.items():
    print(f"  {param:25s}: {desc}")

print("\nReturns:")
print("  input_population: Dict with LGN->V1 connectivity")
print("  network: Dict with V1 recurrent connectivity and parameters")
print("  bkg_weights: Background synaptic weights")

In [ ]:
try:
    # Load network
    print("Loading Billeh network...")
    input_pop, network, bkg_weights = load_billeh(
        n_input=17400,
        n_neurons=10000,  # Subset for faster loading; use 51316 for full
        core_only=True,
        data_dir=str(DATA_DIR),
        seed=3000,
    )
    
    print("\nNetwork loaded successfully!")
    print("=" * 60)
    print(f"Number of V1 neurons: {network['n_nodes']:,}")
    print(f"Number of synapses: {network['n_edges']:,}")
    print(f"Number of neuron types: {len(set(network['node_type_ids']))}")
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please update DATA_DIR.")
    network = None
    input_pop = None

## 4. Neuron Parameters (node_params)

The `node_params` dict contains biophysical parameters for each neuron type.

In [ ]:
if network is not None:
    node_params = network['node_params']
    
    print("Node Parameters")
    print("=" * 60)
    
    param_descriptions = {
        'V_th': ('Threshold voltage', 'mV'),
        'V_reset': ('Reset voltage after spike', 'mV'),
        'E_L': ('Leak reversal potential', 'mV'),
        'g': ('Membrane conductance', 'nS'),
        'C_m': ('Membrane capacitance', 'pF'),
        't_ref': ('Refractory period', 'ms'),
        'tau_syn': ('Synaptic time constants (4 receptors)', 'ms'),
        'k': ('ASC decay rates (2 channels)', '1/ms'),
        'asc_amps': ('ASC amplitudes (2 channels)', 'pA'),
    }
    
    for key, value in node_params.items():
        desc, unit = param_descriptions.get(key, ('', ''))
        if isinstance(value, np.ndarray):
            print(f"\n{key}: {desc}")
            print(f"  Shape: {value.shape}")
            print(f"  Range: [{value.min():.2f}, {value.max():.2f}] {unit}")
            print(f"  Mean: {value.mean():.2f} {unit}")

In [ ]:
if network is not None:
    # Visualize parameter distributions
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    
    params_to_plot = [
        ('V_th', 'Threshold Voltage', 'mV'),
        ('E_L', 'Leak Potential', 'mV'),
        ('g', 'Conductance', 'nS'),
        ('C_m', 'Capacitance', 'pF'),
        ('t_ref', 'Refractory Period', 'ms'),
    ]
    
    for ax, (param, title, unit) in zip(axes.flatten()[:5], params_to_plot):
        values = node_params[param]
        ax.hist(values, bins=30, alpha=0.7, edgecolor='black')
        ax.axvline(x=values.mean(), color='red', linestyle='--', 
                   label=f'Mean: {values.mean():.1f}')
        ax.set_xlabel(f'{param} ({unit})')
        ax.set_ylabel('Count (neuron types)')
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplot
    axes[1, 2].axis('off')
    
    plt.suptitle('Distribution of Neuron Parameters Across Types', fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. Neuron Type Statistics

Analyze the distribution of neuron types in the network.

In [ ]:
if network is not None:
    type_ids = network['node_type_ids']
    type_counts = Counter(type_ids)
    
    print("Neuron Type Statistics")
    print("=" * 60)
    print(f"Total neurons: {len(type_ids):,}")
    print(f"Unique types: {len(type_counts)}")
    
    print(f"\nMost common types:")
    for type_id, count in type_counts.most_common(10):
        print(f"  Type {type_id:3d}: {count:5,} neurons ({100*count/len(type_ids):.1f}%)")
    
    print(f"\nLeast common types:")
    for type_id, count in sorted(type_counts.items(), key=lambda x: x[1])[:5]:
        print(f"  Type {type_id:3d}: {count:5,} neurons ({100*count/len(type_ids):.2f}%)")

In [ ]:
if network is not None:
    # Plot type distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    sorted_counts = sorted(type_counts.items())
    type_indices = [t[0] for t in sorted_counts]
    counts = [t[1] for t in sorted_counts]
    
    axes[0].bar(range(len(type_counts)), counts, alpha=0.7)
    axes[0].set_xlabel('Neuron Type ID')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Neuron Type Distribution')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Histogram of type sizes
    axes[1].hist(counts, bins=30, alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Neurons per Type')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Distribution of Type Sizes')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if network is not None:
    # Neuron spatial distribution
    x = network['x']
    y = network['y']
    z = network['z']
    
    fig = plt.figure(figsize=(14, 5))
    
    # XY projection
    ax1 = fig.add_subplot(131)
    ax1.scatter(x, y, s=0.5, alpha=0.3)
    ax1.set_xlabel('X (um)')
    ax1.set_ylabel('Y (um)')
    ax1.set_title('XY Projection (Horizontal)')
    ax1.set_aspect('equal')
    
    # XZ projection  
    ax2 = fig.add_subplot(132)
    ax2.scatter(x, z, s=0.5, alpha=0.3)
    ax2.set_xlabel('X (um)')
    ax2.set_ylabel('Z (um)')
    ax2.set_title('XZ Projection')
    ax2.set_aspect('equal')
    
    # 3D view
    ax3 = fig.add_subplot(133, projection='3d')
    # Subsample for 3D visualization
    idx = np.random.choice(len(x), min(5000, len(x)), replace=False)
    ax3.scatter(x[idx], y[idx], z[idx], s=0.5, alpha=0.3)
    ax3.set_xlabel('X')
    ax3.set_ylabel('Y')
    ax3.set_zlabel('Z')
    ax3.set_title('3D View (subsampled)')
    
    plt.suptitle('Neuron Spatial Distribution', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Radius from center (core selection)
    r = np.sqrt(x**2 + z**2)
    print(f"\nRadius from center:")
    print(f"  Min: {r.min():.1f} um")
    print(f"  Max: {r.max():.1f} um")
    print(f"  Within 400um: {np.sum(r < 400):,} neurons")

## 6. Input Population

The input population represents LGN neurons that project to V1.

In [ ]:
if input_pop is not None:
    print("Input Population (LGN -> V1)")
    print("=" * 60)
    
    for key, value in input_pop.items():
        if isinstance(value, np.ndarray):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"  {key}: {value}")
    
    print(f"\nInput Statistics:")
    print(f"  Number of LGN inputs: {input_pop['n_inputs']:,}")
    print(f"  Number of LGN->V1 synapses: {len(input_pop['weights']):,}")
    print(f"  Weight range: [{input_pop['weights'].min():.4f}, {input_pop['weights'].max():.4f}]")
    print(f"  Mean weight: {input_pop['weights'].mean():.4f}")

In [ ]:
if input_pop is not None:
    # Visualize input connectivity
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Input weight distribution
    axes[0].hist(input_pop['weights'], bins=50, alpha=0.7, edgecolor='black')
    axes[0].axvline(x=input_pop['weights'].mean(), color='red', linestyle='--',
                    label=f'Mean: {input_pop["weights"].mean():.4f}')
    axes[0].set_xlabel('Weight')
    axes[0].set_ylabel('Count')
    axes[0].set_title('LGN->V1 Weight Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Input degree per V1 neuron
    post_indices = input_pop['indices'][:, 0] // 4  # Remove receptor multiplexing
    in_degree = np.bincount(post_indices, minlength=network['n_nodes'])
    
    axes[1].hist(in_degree, bins=50, alpha=0.7, edgecolor='black')
    axes[1].axvline(x=in_degree.mean(), color='red', linestyle='--',
                    label=f'Mean: {in_degree.mean():.1f}')
    axes[1].set_xlabel('Number of LGN Inputs')
    axes[1].set_ylabel('Count (V1 neurons)')
    axes[1].set_title('LGN Inputs per V1 Neuron')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Background weights
if network is not None:
    print("\nBackground Weights")
    print("=" * 60)
    print(f"Shape: {bkg_weights.shape}")
    print(f"Non-zero: {np.sum(bkg_weights != 0):,}")
    print(f"Range: [{bkg_weights.min():.4f}, {bkg_weights.max():.4f}]")
    
    nonzero_bkg = bkg_weights[bkg_weights != 0]
    if len(nonzero_bkg) > 0:
        print(f"Mean (non-zero): {nonzero_bkg.mean():.4f}")

## 7. Data Validation

Perform sanity checks on the loaded data.

In [ ]:
if network is not None:
    print("Data Validation Checks")
    print("=" * 60)
    
    checks = []
    
    # Check 1: Node count matches
    n_nodes = network['n_nodes']
    pos_count = len(network['x'])
    type_count = len(network['node_type_ids'])
    check1 = (n_nodes == pos_count == type_count)
    checks.append(('Node count consistency', check1, 
                   f'n_nodes={n_nodes}, x={pos_count}, types={type_count}'))
    
    # Check 2: Synapse count matches indices
    n_edges = network['n_edges']
    idx_count = len(network['synapses']['indices'])
    weight_count = len(network['synapses']['weights'])
    check2 = (n_edges == idx_count == weight_count)
    checks.append(('Synapse count consistency', check2,
                   f'n_edges={n_edges}, indices={idx_count}, weights={weight_count}'))
    
    # Check 3: Index ranges valid
    indices = network['synapses']['indices']
    max_post = indices[:, 0].max()
    max_pre = indices[:, 1].max()
    check3 = (max_post < n_nodes * 4) and (max_pre < n_nodes)
    checks.append(('Index ranges valid', check3,
                   f'max_post={max_post} (limit={n_nodes*4}), max_pre={max_pre} (limit={n_nodes})'))
    
    # Check 4: No NaN values
    has_nan = (
        np.any(np.isnan(network['x'])) or
        np.any(np.isnan(network['synapses']['weights']))
    )
    check4 = not has_nan
    checks.append(('No NaN values', check4, 'Checked positions and weights'))
    
    # Check 5: Reasonable parameter ranges
    params = network['node_params']
    check5 = (
        np.all(params['V_th'] > params['E_L']) and  # Threshold above rest
        np.all(params['t_ref'] >= 0) and             # Non-negative refractory
        np.all(params['g'] > 0)                      # Positive conductance
    )
    checks.append(('Reasonable parameter ranges', check5, 'V_th > E_L, t_ref >= 0, g > 0'))
    
    # Print results
    all_passed = True
    for name, passed, details in checks:
        status = "PASS" if passed else "FAIL"
        icon = "OK" if passed else "X"
        print(f"  [{icon}] {name}: {status}")
        if not passed:
            print(f"      Details: {details}")
            all_passed = False
    
    print("\n" + "=" * 60)
    if all_passed:
        print("All validation checks passed!")
    else:
        print("Some checks failed. Please review the data.")

---

## Summary

Key takeaways:

1. **Billeh Network**: Biologically realistic V1 model with ~51K neurons and ~300M synapses

2. **Data Files**:
   - `network_dat.pkl`: Recurrent connectivity
   - `input_dat.pkl`: LGN input connectivity
   - `v1_nodes.h5`: Neuron positions
   - `v1_node_types.csv`: Type definitions

3. **load_billeh()**: Main function returning `(input_pop, network, bkg_weights)`

4. **node_params**: Per-type biophysical parameters (V_th, E_L, g, C_m, etc.)

5. **Neuron Types**: 111 distinct types with different parameters

6. **Input Population**: LGN->V1 connectivity with ~17K input neurons

---

## Exercises

1. **Load full network**: Set `n_neurons=51316` and explore the full connectivity.

2. **Layer analysis**: Use y-coordinate to identify cortical layers.

3. **E/I classification**: Use node type info to classify neurons as E or I.

In [ ]:
# Exercise starter: Classify E/I neurons
try:
    df = pd.read_csv(DATA_DIR / "network" / "v1_node_types.csv", delimiter=' ')
    
    # Count excitatory vs inhibitory types
    exc_types = df[df['pop_name'].str.startswith('e')]
    inh_types = df[df['pop_name'].str.startswith('i')]
    
    print(f"Excitatory neuron types: {len(exc_types)}")
    print(f"Inhibitory neuron types: {len(inh_types)}")
    
    print("\nTODO: Map type IDs to neurons and compute E/I ratio in the network")
except FileNotFoundError:
    print("Node types file not found.")

---

## Source Files

- `src/v1_jax/data/network_loader.py` - Network loading utilities

## References

- Billeh et al., "Systematic Integration of Structural and Functional Data into Multi-scale Models of Mouse Primary Visual Cortex", Neuron 2020
- Allen Institute V1 Model: https://portal.brain-map.org/explore/models/mv1-all-layers